In [ ]:
import os
import shutil
import random
import numpy as np
from PIL import Image, ImageEnhance, ImageFilter

# ==========================
# Source and Destination
# ==========================

SOURCE_DIR = r"D:\review\work\"
SAVE_DIR = r"D:\review\work\guava640640"

os.makedirs(SAVE_DIR, exist_ok=True)

# ==========================
# Classes
# ==========================

CLASSES = [
    "algal_leaf_spot",
    "fruit_black_mold",
    "fruit_healthy",
    "fruit_scab",
    "healthy_leaf",
    "insect_bite_leaf",
    "mealybug_leaf",
    "red_rust_leaf",
    "scorch_leaf",
    "yellow_leaf_disease"
]

# ==========================
# Helper Function
# ==========================

def save_augmented(img, label_path, save_img_path, save_label_path):
    img.save(save_img_path, quality=100)
    shutil.copy2(label_path, save_label_path)

# ==========================
# Processing
# ==========================

for cls in CLASSES:

    print(f"\nProcessing: {cls}")

    src_img_dir = os.path.join(SOURCE_DIR, cls, "images")
    src_lbl_dir = os.path.join(SOURCE_DIR, cls, "labels")

    dst_img_dir = os.path.join(SAVE_DIR, cls, "images")
    dst_lbl_dir = os.path.join(SAVE_DIR, cls, "labels")

    os.makedirs(dst_img_dir, exist_ok=True)
    os.makedirs(dst_lbl_dir, exist_ok=True)

    image_files = [
        f for f in os.listdir(src_img_dir)
        if f.lower().endswith((".jpg", ".jpeg", ".png"))
    ]

    for image_file in image_files:

        image_path = os.path.join(src_img_dir, image_file)

        label_file = os.path.splitext(image_file)[0] + ".txt"
        label_path = os.path.join(src_lbl_dir, label_file)

        if not os.path.exists(label_path):
            continue

        img = Image.open(image_path).convert("RGB")

        base_name = os.path.splitext(image_file)[0]

        # --------------------------------
        # Copy Original
        # --------------------------------

        shutil.copy2(
            image_path,
            os.path.join(dst_img_dir, image_file)
        )

        shutil.copy2(
            label_path,
            os.path.join(dst_lbl_dir, label_file)
        )

        # --------------------------------
        # Horizontal Flip
        # --------------------------------

        save_augmented(
            img.transpose(Image.FLIP_LEFT_RIGHT),
            label_path,
            os.path.join(dst_img_dir, f"{base_name}_hflip.jpg"),
            os.path.join(dst_lbl_dir, f"{base_name}_hflip.txt")
        )

        # --------------------------------
        # Vertical Flip
        # --------------------------------

        save_augmented(
            img.transpose(Image.FLIP_TOP_BOTTOM),
            label_path,
            os.path.join(dst_img_dir, f"{base_name}_vflip.jpg"),
            os.path.join(dst_lbl_dir, f"{base_name}_vflip.txt")
        )

        # --------------------------------
        # Rotate 90
        # --------------------------------

        save_augmented(
            img.rotate(90, expand=True),
            label_path,
            os.path.join(dst_img_dir, f"{base_name}_rotate90.jpg"),
            os.path.join(dst_lbl_dir, f"{base_name}_rotate90.txt")
        )

        # --------------------------------
        # Rotate 180
        # --------------------------------

        save_augmented(
            img.rotate(180, expand=True),
            label_path,
            os.path.join(dst_img_dir, f"{base_name}_rotate180.jpg"),
            os.path.join(dst_lbl_dir, f"{base_name}_rotate180.txt")
        )

        # --------------------------------
        # Shear
        # --------------------------------

        save_augmented(
            img.transform(
                img.size,
                Image.AFFINE,
                (1, 0.3, 0, 0, 1, 0)
            ),
            label_path,
            os.path.join(dst_img_dir, f"{base_name}_shear.jpg"),
            os.path.join(dst_lbl_dir, f"{base_name}_shear.txt")
        )

        # --------------------------------
        # Brightness
        # --------------------------------

        save_augmented(
            ImageEnhance.Brightness(img).enhance(1.5),
            label_path,
            os.path.join(dst_img_dir, f"{base_name}_brightness.jpg"),
            os.path.join(dst_lbl_dir, f"{base_name}_brightness.txt")
        )

        # --------------------------------
        # Color Enhancement
        # --------------------------------

        save_augmented(
            ImageEnhance.Color(img).enhance(1.5),
            label_path,
            os.path.join(dst_img_dir, f"{base_name}_color.jpg"),
            os.path.join(dst_lbl_dir, f"{base_name}_color.txt")
        )

        # --------------------------------
        # Contrast
        # --------------------------------

        save_augmented(
            ImageEnhance.Contrast(img).enhance(1.5),
            label_path,
            os.path.join(dst_img_dir, f"{base_name}_contrast.jpg"),
            os.path.join(dst_lbl_dir, f"{base_name}_contrast.txt")
        )

        # --------------------------------
        # Gamma
        # --------------------------------

        gamma = 0.7
        gamma_img = np.array(img) / 255.0
        gamma_img = np.power(gamma_img, gamma)
        gamma_img = np.uint8(gamma_img * 255)

        save_augmented(
            Image.fromarray(gamma_img),
            label_path,
            os.path.join(dst_img_dir, f"{base_name}_gamma.jpg"),
            os.path.join(dst_lbl_dir, f"{base_name}_gamma.txt")
        )

        # --------------------------------
        # Color Jitter
        # --------------------------------

        jitter = ImageEnhance.Brightness(img).enhance(
            random.uniform(0.7, 1.3)
        )
        jitter = ImageEnhance.Contrast(jitter).enhance(
            random.uniform(0.7, 1.3)
        )
        jitter = ImageEnhance.Color(jitter).enhance(
            random.uniform(0.7, 1.3)
        )

        save_augmented(
            jitter,
            label_path,
            os.path.join(dst_img_dir, f"{base_name}_colorjitter.jpg"),
            os.path.join(dst_lbl_dir, f"{base_name}_colorjitter.txt")
        )

        # --------------------------------
        # Channel Shuffle
        # --------------------------------

        arr = np.array(img)
        channels = [0, 1, 2]
        random.shuffle(channels)

        save_augmented(
            Image.fromarray(arr[:, :, channels]),
            label_path,
            os.path.join(dst_img_dir, f"{base_name}_channelshuffle.jpg"),
            os.path.join(dst_lbl_dir, f"{base_name}_channelshuffle.txt")
        )

        # --------------------------------
        # Grayscale
        # --------------------------------

        save_augmented(
            img.convert("L").convert("RGB"),
            label_path,
            os.path.join(dst_img_dir, f"{base_name}_grayscale.jpg"),
            os.path.join(dst_lbl_dir, f"{base_name}_grayscale.txt")
        )

        # --------------------------------
        # Gaussian Blur
        # --------------------------------

        save_augmented(
            img.filter(ImageFilter.GaussianBlur(radius=2)),
            label_path,
            os.path.join(dst_img_dir, f"{base_name}_blur.jpg"),
            os.path.join(dst_lbl_dir, f"{base_name}_blur.txt")
        )

        # --------------------------------
        # Gaussian Noise
        # --------------------------------

        noise = np.random.normal(
            0,
            20,
            np.array(img).shape
        )

        noisy = np.clip(
            np.array(img) + noise,
            0,
            255
        ).astype(np.uint8)

        save_augmented(
            Image.fromarray(noisy),
            label_path,
            os.path.join(dst_img_dir, f"{base_name}_gaussiannoise.jpg"),
            os.path.join(dst_lbl_dir, f"{base_name}_gaussiannoise.txt")
        )

        # --------------------------------
        # Salt & Pepper
        # --------------------------------

        sp = np.array(img).copy()

        prob = 0.02

        salt = np.random.random(sp.shape[:2]) < prob
        pepper = np.random.random(sp.shape[:2]) < prob

        sp[salt] = 255
        sp[pepper] = 0

        save_augmented(
            Image.fromarray(sp),
            label_path,
            os.path.join(dst_img_dir, f"{base_name}_saltpepper.jpg"),
            os.path.join(dst_lbl_dir, f"{base_name}_saltpepper.txt")
        )

    print(f"{cls} completed.")

print("\nAll augmentations completed successfully.")

In [3]:
import os

folder_path = r"D:\review\Version4_Raw_Metadata_Label_Split"

if os.path.exists(folder_path):
    print(f"Root Folder: {folder_path}\n")

    for root, dirs, files in os.walk(folder_path):
        print(f"\nCurrent Folder: {root}")

        if dirs:
            print("Subfolders:")
            for d in dirs:
                print(f"  [DIR]  {d}")

        if files:
            print("Files:")
            for f in files:
                print(f"  [FILE] {f}")
else:
    print("Folder does not exist.")

Root Folder: D:\review\Version4_Raw_Metadata_Label_Split


Current Folder: D:\review\Version4_Raw_Metadata_Label_Split
Subfolders:
  [DIR]  augmentation_split_t_t_v
  [DIR]  label_class_yolov8
  [DIR]  original_raw_data
  [DIR]  resized_640-640
Files:
  [FILE] augmentation_config.py
  [FILE] Data_Collection_Time.png
  [FILE] metadata.csv

Current Folder: D:\review\Version4_Raw_Metadata_Label_Split\augmentation_split_t_t_v
Subfolders:
  [DIR]  test
  [DIR]  train
  [DIR]  valid

Current Folder: D:\review\Version4_Raw_Metadata_Label_Split\augmentation_split_t_t_v\test
Subfolders:
  [DIR]  algal_leaf_spot
  [DIR]  fruit_black_mold
  [DIR]  fruit_healthy
  [DIR]  fruit_scab
  [DIR]  healthy_leaf
  [DIR]  insect_bite_leaf
  [DIR]  mealybug_leaf
  [DIR]  red_rust_leaf
  [DIR]  scorch_leaf
  [DIR]  yellow_leaf_disease

Current Folder: D:\review\Version4_Raw_Metadata_Label_Split\augmentation_split_t_t_v\test\algal_leaf_spot
Subfolders:
  [DIR]  images
  [DIR]  labels

Current Folder: D:\review

In [ ]:
Root Folder: D:\review\Version4_Raw_Metadata_Label_Split


Current Folder: D:\review\Version4_Raw_Metadata_Label_Split
Subfolders:
  [DIR]  augmentation_split_t_t_v
  [DIR]  label_class_yolov8
  [DIR]  original_raw_data
  [DIR]  resized_640-640
Files:
  [FILE] augmentation_config.py
 
  [FILE] metadata.csv

Current Folder: D:\review\Version4_Raw_Metadata_Label_Split\augmentation_split_t_t_v
Subfolders:
  [DIR]  test
  [DIR]  train
  [DIR]  valid

Current Folder: D:\review\Version4_Raw_Metadata_Label_Split\augmentation_split_t_t_v\test
Subfolders:
  [DIR]  algal_leaf_spot
  [DIR]  fruit_black_mold
  [DIR]  fruit_healthy
  [DIR]  fruit_scab
  [DIR]  healthy_leaf
  [DIR]  insect_bite_leaf
  [DIR]  mealybug_leaf
  [DIR]  red_rust_leaf
  [DIR]  scorch_leaf
  [DIR]  yellow_leaf_disease

Current Folder: D:\review\Version4_Raw_Metadata_Label_Split\augmentation_split_t_t_v\test\algal_leaf_spot
Subfolders:
  [DIR]  images
  [DIR]  labels

Current Folder: D:\review\Version4_Raw_Metadata_Label_Split\augmentation_split_t_t_v\test\algal_leaf_spot\images
Files:
  [FILE] algal_leaf_spot_003.jpg
 

In [4]:
import os

root_folder = r"D:\review\Version4_Raw_Metadata_Label_Split"

for folder_name in os.listdir(root_folder):
    folder_path = os.path.join(root_folder, folder_name)

    if os.path.isdir(folder_path):
        file_count = sum(
            1 for item in os.listdir(folder_path)
            if os.path.isfile(os.path.join(folder_path, item))
        )

        print(f"{folder_name}: {file_count} files")

augmentation_split_t_t_v: 0 files
label_class_yolov8: 1 files
original_raw_data: 0 files
resized_640-640: 0 files
